# nb_get_last_partition

Called by ADF pipeline `pl_bronze_blob_charging_sessions` at the start of every incremental run.

Queries `dbw_ev_intelligence_dev.default.bronze_blob_audit` for the last successful partition copied for this pipeline, then advances it by one hour. Returns the next partition as a JSON string via `dbutils.notebook.exit()` — ADF reads this from `.output.runOutput`.

**If no audit record exists** (first ever run): returns the `FALLBACK_PARTITION` defined in Cell 1.

**Hour advancement logic:**
- Last hour `< 23` → same day, hour + 1
- Last hour `= 23` → next day (midnight rollover), handles month/year boundaries

**Output format** (consumed by ADF `act_get_last_partition`):
```json
{"year": "2026", "month": "06", "day": "01", "hour": "07"}
```

## Cell 1 — Parameters and fallback partition

ADF passes `pipeline_name` as a notebook parameter via `baseParameters`. Databricks widgets receive it.

`FALLBACK_PARTITION` is only used when no audit record exists — i.e., the very first run. Set it to the earliest partition you want to ingest.

In [ ]:
dbutils.widgets.text("pipeline_name", "pl_bronze_blob_charging_sessions")

PIPELINE_NAME      = dbutils.widgets.get("pipeline_name")
AUDIT_TABLE        = "dbw_ev_intelligence_dev.default.bronze_blob_audit"
FALLBACK_PARTITION = {"year": "2026", "month": "06", "day": "01", "hour": "06"}

print(f"Pipeline      : {PIPELINE_NAME}")
print(f"Audit table   : {AUDIT_TABLE}")
print(f"Fallback      : {FALLBACK_PARTITION}")

## Cell 2 — Query audit table for last successful partition

Reads the most recent row from the audit table where `pipeline_name` matches and `status = 'succeeded'`.

- `spark.catalog.tableExists` safely checks before querying — avoids an error on the first run when the table does not yet exist
- Orders by `run_timestamp desc` and takes 1 row — most recent successful run
- If the table exists but has no succeeded rows yet, `last_row` will be `None` and the fallback partition is used

In [ ]:
import json

last_row = None

if spark.catalog.tableExists(AUDIT_TABLE):
    df_audit = spark.sql(f"""
        SELECT p_year, p_month, p_day, p_hour
        FROM   {AUDIT_TABLE}
        WHERE  pipeline_name = '{PIPELINE_NAME}'
          AND  status        = 'succeeded'
        ORDER BY run_timestamp DESC
        LIMIT 1
    """)
    rows = df_audit.collect()
    if rows:
        last_row = rows[0]

if last_row:
    print(f"Last successful partition: {last_row.p_year}/{last_row.p_month}/{last_row.p_day}/{last_row.p_hour}")
else:
    print("No audit record found — will use fallback partition")

## Cell 3 — Advance partition by one hour

Takes the last successfully copied partition and returns the **next** hour as the target for this run.

Uses Python `datetime` for correct rollover:
- Hour 23 of any day → rolls to hour 00 of the next day
- Last day of a month → rolls to day 01 of the next month
- December 31 hour 23 → rolls to January 01 of the next year

All values are zero-padded to 2 digits to match the source blob folder names exactly.

In [ ]:
from datetime import datetime, timedelta

if last_row:
    last_dt = datetime(
        int(last_row.p_year),
        int(last_row.p_month),
        int(last_row.p_day),
        int(last_row.p_hour)
    )
    next_dt = last_dt + timedelta(hours=1)
    next_partition = {
        "year":  next_dt.strftime("%Y"),
        "month": next_dt.strftime("%m"),
        "day":   next_dt.strftime("%d"),
        "hour":  next_dt.strftime("%H")
    }
else:
    next_partition = FALLBACK_PARTITION

print(f"Next partition to copy: {next_partition['year']}/{next_partition['month']}/{next_partition['day']}/{next_partition['hour']}")

## Cell 4 — Return result to ADF

`dbutils.notebook.exit()` passes a string back to the ADF pipeline. ADF reads it via:
```
activity('act_get_last_partition').output.runOutput
```

The JSON string is then parsed in ADF using `@json(...)` to extract individual year/month/day/hour values into pipeline variables.

In [ ]:
result = json.dumps(next_partition)
print(f"Returning to ADF: {result}")
dbutils.notebook.exit(result)